# ASTER — DLPFC Real Data

Applies ASTER to all 12 DLPFC slices with one or more random seeds.
Outputs per-slice `.h5ad` files and a summary CSV with ARI scores.

## Data you need to download first

This repository ships the data directories **empty**. Download package
**`<DLPFC_URL>`** and unpack it into `preprocess_data/dlpfc/` (see the README there):
24 files, ~867 MB -- one `*_preprocessed.h5ad` plus one `*_truth.txt` per slice, for the
12 DLPFC slices. These are the benchmark-ready inputs, so no upstream preprocessing is
needed.

The `*_truth.txt` annotations are only used for the ARI evaluation. This notebook also
needs a system R with `mclust`; set `R_HOME` in the next cell accordingly.

Check your download before running the rest:

In [ ]:
import subprocess, sys

from repro_st_aster.common import find_repo_root

REPO_ROOT = find_repo_root()
subprocess.run([sys.executable, str(REPO_ROOT / 'scripts' / 'check_data.py'), 'dlpfc'])

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import anndata
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from tqdm import tqdm

from repro_st_aster.aster_ntd import ASTER, load_dlpfc_slice
from repro_st_aster.common import find_repo_root

os.environ.setdefault('R_HOME', '/path/to/R')
import rpy2.robjects as robjects
import rpy2.robjects.numpy2ri
from rpy2.robjects.packages import importr

robjects.r.library('mclust')
rpy2.robjects.numpy2ri.activate()
Mclust = robjects.r['Mclust']
r_base = importr('base')


In [ ]:
REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / 'preprocess_data' / 'dlpfc'
OUTPUT_DIR = REPO_ROOT / 'results' / 'dlpfc_notebook'

SEEDS = [2023]
RANK_X = 64
RANK_Y = 64
RANK_G = 256
MID_FEATURE = 200
LR = 0.001
MAX_EPOCH = 1500
N_CLUSTERS = 7
N_PCA = 15

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
slice_files = sorted([f.name for f in DATA_DIR.glob('*_preprocessed.h5ad')])
slice_ids = [int(f.split('_')[0]) for f in slice_files]
print(f'Found {len(slice_ids)} slices: {slice_ids}')


In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False
    r_base.set_seed(seed)

In [ ]:
all_results = []

for seed in tqdm(SEEDS, desc='Seeds'):
    seed_dir = OUTPUT_DIR / f'Seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    set_seed(seed)

    for slice_id in tqdm(slice_ids, desc=f'Seed {seed}', leave=False):
        try:
            # ── 1. Load data ─────────────────────────────────────────────────
            h5ad_path = DATA_DIR / f'{slice_id}_preprocessed.h5ad'
            truth_path = DATA_DIR / f'{slice_id}_truth.txt'
            data = load_dlpfc_slice(h5ad_path, truth_path, use_all_entries=True)

            # ── 2. Fit ASTER ──────────────────────────────────────────────────
            model = ASTER(
                expr_tensor    = data['expr_tensor'],
                gene_names     = data['gene_names'],
                n_x            = data['n_x'],
                n_y            = data['n_y'],
                coords_mapping = data['coords_mapping'],
            )
            model.fit(
                rank_x=RANK_X, rank_y=RANK_Y, rank_g=RANK_G,
                mid_feature=MID_FEATURE, lr=LR, max_epoch=MAX_EPOCH,
                save_dir=str(seed_dir / f'slice_{slice_id}'), verbose=False,
            )

            # ── 3. Imputed matrix ─────────────────────────────────────────────
            expr_imputed, gene_names = model.get_imputed_matrix()

            # ── 4. PCA ───────────────────────────────────────────────────────
            pca = PCA(n_components=N_PCA, random_state=seed)
            expr_pca = pca.fit_transform(expr_imputed)

            # ── 5. Mclust clustering ──────────────────────────────────────────
            mclust_res  = Mclust(rpy2.robjects.numpy2ri.numpy2rpy(expr_pca), N_CLUSTERS, 'EEE')
            pred_labels = np.array(mclust_res[-2]).astype(int)

            # ── 6. ARI ────────────────────────────────────────────────────────
            gt   = pd.Series(data['labels_gt']).astype(str)
            mask = ~gt.isna() & (gt != 'nan')
            ari  = adjusted_rand_score(gt[mask], pred_labels[mask])

            # ── 7. Save .h5ad ─────────────────────────────────────────────────
            adata_out = anndata.AnnData(X=expr_imputed)
            adata_out.obs['ground_truth'] = data['labels_gt']
            adata_out.obs['pred_labels']  = pred_labels.astype(str)
            adata_out.obsm['spatial']     = data['spatial_coords']
            adata_out.obsm['X_pca']       = expr_pca
            adata_out.var_names           = gene_names
            adata_out.uns['seed'] = seed
            adata_out.uns['ARI']          = ari
            adata_out.write(os.path.join(seed_dir, f'Slice_{slice_id}_Seed_{seed}.h5ad'))

            all_results.append({'Slice_ID': slice_id, 'Seed': seed, 'ARI': ari, 'Status': 'Success'})
            model.clear_gpu()

        except Exception as e:
            print(f'[Error] Seed {seed} Slice {slice_id}: {e}')
            all_results.append({'Slice_ID': slice_id, 'Seed': seed, 'ARI': np.nan, 'Status': str(e)})

In [ ]:
df = pd.DataFrame(all_results)
report_path = OUTPUT_DIR / 'DLPFC_ASTER_ARI_summary.csv'
df.to_csv(report_path, index=False)

print(df.pivot_table(values='ARI', index='Slice_ID', columns='Seed'))
print(f'\nMean ARI: {df["ARI"].mean():.4f} ± {df["ARI"].std():.4f}')
print(f'Saved: {report_path}')